# Telco Customer Churn — Exploratory Data Analysis

Before building any model, understand the data. This notebook answers:
- How imbalanced is the target?
- Which features correlate most strongly with churn?
- Which customer segments are highest-risk?

These findings directly motivated the feature engineering in `prepare.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

df = pd.read_csv("data/telco.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)
print(df.shape)
df.head(3)

## 1. Class imbalance — the core problem


In [ ]:
churn_rate = df["Churn"].mean()
counts = df["Churn"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(["Stay", "Churn"], counts.values, color=["#2563eb", "#dc2626"])
axes[0].set_title("Customer counts")
axes[0].set_ylabel("Customers")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha="center", fontweight="bold")

axes[1].pie([counts[0], counts[1]], labels=["Stay", "Churn"],
            colors=["#2563eb", "#dc2626"], autopct="%1.1f%%", startangle=90)
axes[1].set_title(f"Churn rate: {churn_rate:.1%}")

plt.suptitle("Target imbalance: 3 stayers for every 1 churner", fontsize=13)
plt.tight_layout()
plt.show()
print(f"A naive 'always stay' model would be {1-churn_rate:.1%} accurate — but useless.")

## 2. Churn by contract type

Month-to-month customers are the biggest risk — they can leave any time.


In [ ]:
contract_churn = df.groupby("Contract")["Churn"].mean().sort_values(ascending=False)
contract_count = df.groupby("Contract")["Churn"].count()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(contract_churn.index, contract_churn.values, color=["#dc2626", "#f59e0b", "#16a34a"])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Churn rate by contract type", fontsize=13)
ax.set_ylabel("Churn rate")
ax.axhline(df["Churn"].mean(), ls="--", color="grey", label=f"Overall avg {df['Churn'].mean():.1%}")
ax.legend()
for bar, (name, n) in zip(bars, contract_count.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"n={n}", ha="center", fontsize=9, color="grey")
plt.tight_layout()
plt.show()
print(contract_churn.to_string())

## 3. Churn by tenure

New customers churn far more. This motivated the `TenureGroup` and `HighValueNewCustomer` features.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram: tenure by churn status
for label, color, churn_val in [("Stay", "#2563eb", 0), ("Churn", "#dc2626", 1)]:
    axes[0].hist(df[df["Churn"] == churn_val]["tenure"], bins=30,
                 alpha=0.6, label=label, color=color)
axes[0].set_xlabel("Tenure (months)")
axes[0].set_ylabel("Customers")
axes[0].set_title("Tenure distribution by churn")
axes[0].legend()

# Churn rate per tenure bucket
df["TenureGroup"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 1000],
                            labels=["0-1yr", "1-2yr", "2-4yr", "4yr+"])
churn_by_tenure = df.groupby("TenureGroup", observed=True)["Churn"].mean()
axes[1].bar(churn_by_tenure.index, churn_by_tenure.values, color="#6366f1")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title("Churn rate by tenure group")
axes[1].set_ylabel("Churn rate")

plt.suptitle("Newer customers churn significantly more", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Monthly charges vs churn

Higher-paying customers tend to churn more — possibly price-sensitive or over-subscribed.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, val in [("Stay", "#2563eb", 0), ("Churn", "#dc2626", 1)]:
    data = df[df["Churn"] == val]["MonthlyCharges"]
    axes[0].hist(data, bins=30, alpha=0.6, label=f"{label} (n={len(data)})", color=color)
axes[0].set_xlabel("Monthly charges ($)")
axes[0].set_title("Monthly charges by churn")
axes[0].legend()

# Box plot
stay_data  = df[df["Churn"] == 0]["MonthlyCharges"]
churn_data = df[df["Churn"] == 1]["MonthlyCharges"]
axes[1].boxplot([stay_data, churn_data], labels=["Stay", "Churn"],
                patch_artist=True,
                boxprops=dict(facecolor="#dbeafe"),
                medianprops=dict(color="#dc2626", lw=2))
axes[1].set_ylabel("Monthly charges ($)")
axes[1].set_title(f"Median: Stay=${stay_data.median():.0f}  Churn=${churn_data.median():.0f}")

plt.suptitle("Churners pay more on average — price sensitivity", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Internet service, payment method, and other categoricals


In [ ]:
cat_cols = ["InternetService", "PaymentMethod", "TechSupport", "OnlineSecurity"]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    rates = df.groupby(col)["Churn"].mean().sort_values(ascending=False)
    colors = ["#dc2626" if r > df["Churn"].mean() else "#2563eb" for r in rates.values]
    ax.barh(rates.index, rates.values, color=colors)
    ax.axvline(df["Churn"].mean(), ls="--", color="grey", lw=1)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title(f"Churn rate by {col}")

plt.suptitle("Red = above-average churn rate", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Numeric feature correlation with churn


In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
corr = df[numeric_cols + ["Churn"]].corr()["Churn"].drop("Churn").sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#dc2626" if v > 0 else "#2563eb" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Pearson correlation with Churn", fontsize=13)
ax.set_xlabel("Correlation")
plt.tight_layout()
plt.show()
print(corr.to_string())

## 7. Engineered features preview

These are the three features added in `prepare.py` — checking they separate churn classes before committing to them.


In [ ]:
df["AvgChargesPerMonth"]   = df["TotalCharges"] / (df["tenure"] + 1)
df["IsMonthToMonth"]       = (df["Contract"] == "Month-to-month").astype(int)
monthly_median = df["MonthlyCharges"].median()
df["HighValueNewCustomer"] = ((df["tenure"] < 12) & (df["MonthlyCharges"] > monthly_median)).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# AvgChargesPerMonth by churn
for label, val, color in [("Stay", 0, "#2563eb"), ("Churn", 1, "#dc2626")]:
    axes[0].hist(df[df["Churn"]==val]["AvgChargesPerMonth"].clip(0, 120),
                 bins=30, alpha=0.6, label=label, color=color)
axes[0].set_title("AvgChargesPerMonth")
axes[0].legend()

# IsMonthToMonth churn rate
mtm_rate = df.groupby("IsMonthToMonth")["Churn"].mean()
axes[1].bar(["Other contract", "Month-to-month"], mtm_rate.values, color=["#2563eb", "#dc2626"])
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title("IsMonthToMonth churn rate")

# HighValueNewCustomer churn rate
hvnc_rate = df.groupby("HighValueNewCustomer")["Churn"].mean()
axes[2].bar(["Other", "High-value new"], hvnc_rate.values, color=["#2563eb", "#dc2626"])
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[2].set_title("HighValueNewCustomer churn rate")

plt.suptitle("All three engineered features separate churn classes", fontsize=13)
plt.tight_layout()
plt.show()

## Key takeaways

| Finding | Feature engineering response |
|---|---|
| Month-to-month contracts churn at ~42% vs ~11% for annual | `IsMonthToMonth` flag |
| New customers (0-1yr) churn at ~47% | `TenureGroup` buckets |
| High charges + short tenure = price-shocked customers | `HighValueNewCustomer` |
| Churners pay ~$15/mo more on average | `AvgChargesPerMonth` |
| Fiber optic users churn more than DSL | Already captured in one-hot encoding |
| Electronic check payers churn significantly more | Already captured in one-hot encoding |

**Next step:** run `python prepare.py` to apply these transformations, then `python train.py` to build and evaluate the models.
